In [1]:
pip install pandas openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\programdata\anaconda3\lib\site-packages\vboxapi-1.0-py3.11.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..


In [6]:
import os
import json
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

GENOME_DIR = r"C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data"

PROJECT_DIR = r"C:\Users\HP\Downloads\klebsiella_project"

EXCEL_FILE = os.path.join(
    PROJECT_DIR,
    "mgen-9-967-s002.xlsx"
)

JSON_FILE = os.path.join(
    GENOME_DIR,
    "assembly_data_report.jsonl"
)


# ============================================================
# OUTPUT FILES
# ============================================================

MAPPING_OUTPUT = os.path.join(
    PROJECT_DIR,
    "genome_MRSN_Cefepime_mapping.csv"
)

RS_OUTPUT = os.path.join(
    PROJECT_DIR,
    "genome_MRSN_Cefepime_RS_only.csv"
)

UNMATCHED_AST_OUTPUT = os.path.join(
    PROJECT_DIR,
    "AST_without_NCBI_genome.csv"
)


# ============================================================
# CHECK FILES
# ============================================================

print("=" * 70)
print("CHECKING FILES")
print("=" * 70)

if not os.path.exists(JSON_FILE):
    raise FileNotFoundError(
        f"NCBI metadata file not found:\n{JSON_FILE}"
    )

if not os.path.exists(EXCEL_FILE):
    raise FileNotFoundError(
        f"Excel file not found:\n{EXCEL_FILE}"
    )

print("✓ NCBI metadata file found")
print("✓ Excel AST file found")


# ============================================================
# STEP 1 — READ NCBI JSONL
# ============================================================

print("\n" + "=" * 70)
print("STEP 1 — READING NCBI METADATA")
print("=" * 70)

records = []

with open(
    JSON_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        accession = record.get(
            "accession"
        )

        assembly_info = record.get(
            "assemblyInfo",
            {}
        )

        biosample = assembly_info.get(
            "biosample",
            {}
        )

        biosample_id = biosample.get(
            "accession"
        )

        # ----------------------------------------------------
        # Get MRSN from organism strain
        # ----------------------------------------------------

        organism = record.get(
            "organism",
            {}
        )

        infraspecific_names = organism.get(
            "infraspecificNames",
            {}
        )

        mrsn = infraspecific_names.get(
            "strain"
        )

        # ----------------------------------------------------
        # Backup: search BioSample attributes
        # ----------------------------------------------------

        if not mrsn:

            attributes = biosample.get(
                "attributes",
                []
            )

            for attribute in attributes:

                name = str(
                    attribute.get(
                        "name",
                        ""
                    )
                ).lower()

                value = attribute.get(
                    "value"
                )

                if (
                    "strain" in name
                    and value
                ):
                    mrsn = str(value)
                    break

        records.append({
            "NCBI_Accession": accession,
            "BioSample": biosample_id,
            "MRSN": mrsn
        })


genome_df = pd.DataFrame(
    records
)


print(
    "NCBI metadata records:",
    len(genome_df)
)

print("\nFirst 10 records:")

print(
    genome_df.head(10).to_string(
        index=False
    )
)


# ============================================================
# STEP 2 — CLEAN NCBI MRSN
# ============================================================

genome_df["NCBI_Accession"] = (
    genome_df["NCBI_Accession"]
    .astype(str)
    .str.strip()
)

genome_df["BioSample"] = (
    genome_df["BioSample"]
    .astype(str)
    .str.strip()
)

genome_df["MRSN"] = (
    genome_df["MRSN"]
    .astype(str)
    .str.strip()
)


# ============================================================
# STEP 3 — CHECK DOWNLOADED GENOMES
# ============================================================

print("\n" + "=" * 70)
print("STEP 2 — CHECKING DOWNLOADED GENOMES")
print("=" * 70)

gca_folders = []

for item in os.listdir(GENOME_DIR):

    full_path = os.path.join(
        GENOME_DIR,
        item
    )

    if (
        os.path.isdir(full_path)
        and item.startswith("GCA_")
    ):
        gca_folders.append(item)


print(
    "Downloaded GCA genome folders:",
    len(gca_folders)
)

print("\nFirst 10 downloaded genomes:")

for genome in gca_folders[:10]:
    print(genome)


# ============================================================
# STEP 4 — READ EXCEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — READING AST EXCEL")
print("=" * 70)

raw = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Table S1",
    header=None
)

print(
    "Excel dimensions:",
    raw.shape
)


# ============================================================
# KNOWN EXCEL POSITIONS
# ============================================================

HEADER_ROW = 1

MRSN_COLUMN = 0

FEP_COLUMN = 24


print("\nExcel header information:")

print(
    "Header row      :",
    HEADER_ROW
)

print(
    "MRSN column     :",
    MRSN_COLUMN
)

print(
    "Cefepime column :",
    FEP_COLUMN
)

print(
    "MRSN header     :",
    raw.iloc[
        HEADER_ROW,
        MRSN_COLUMN
    ]
)

print(
    "FEP header      :",
    raw.iloc[
        HEADER_ROW,
        FEP_COLUMN
    ]
)


# ============================================================
# STEP 5 — EXTRACT AST DATA
# ============================================================

ast_df = raw.iloc[
    HEADER_ROW + 1:
].copy()

ast_df = ast_df[
    [
        MRSN_COLUMN,
        FEP_COLUMN
    ]
].copy()

ast_df.columns = [
    "MRSN",
    "Cefepime"
]


# ============================================================
# STEP 6 — FIX MRSN IDENTIFIERS
# ============================================================

print("\n" + "=" * 70)
print("STEP 4 — CONVERTING EXCEL MRSN IDs")
print("=" * 70)


def convert_mrsn(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    # Remove decimal .0 if Excel interpreted
    # the number as a float.
    if value.endswith(".0"):
        value = value[:-2]

    # Already has MRSN prefix
    if value.upper().startswith("MRSN"):

        return value.upper()

    # Numeric Excel value
    if value.isdigit():

        return "MRSN" + value

    return None


ast_df["MRSN"] = (
    ast_df["MRSN"]
    .apply(convert_mrsn)
)


# ============================================================
# STEP 7 — CLEAN CEFEPIME
# ============================================================

ast_df["Cefepime"] = (
    ast_df["Cefepime"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# STEP 8 — NORMALIZE AST VALUES
# ============================================================

ast_df["Cefepime"] = (
    ast_df["Cefepime"]
    .replace({
        "SUSCEPTIBLE": "S",
        "SENSITIVE": "S",
        "RESISTANT": "R",
        "INTERMEDIATE": "I"
    })
)


# ============================================================
# STEP 9 — REMOVE INVALID ROWS
# ============================================================

ast_df = ast_df[
    ast_df["MRSN"].notna()
].copy()


# ============================================================
# STEP 10 — DISPLAY AST
# ============================================================

print("\nFirst 20 AST records:")

print(
    ast_df.head(20).to_string(
        index=False
    )
)

print(
    "\nAST records found:",
    len(ast_df)
)

print(
    "\nCefepime values:"
)

print(
    ast_df[
        "Cefepime"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# STEP 11 — REMOVE DUPLICATES
# ============================================================

duplicates = (
    ast_df["MRSN"]
    .duplicated()
    .sum()
)

print(
    "\nDuplicate MRSN records:",
    duplicates
)

ast_df = ast_df.drop_duplicates(
    subset="MRSN",
    keep="first"
)

print(
    "AST records after duplicate removal:",
    len(ast_df)
)


# ============================================================
# STEP 12 — SHOW EXAMPLE MRSN MATCHING
# ============================================================

print("\n" + "=" * 70)
print("STEP 5 — CHECKING MRSN FORMAT")
print("=" * 70)

print("\nExcel converted MRSN IDs:")

print(
    ast_df["MRSN"]
    .head(10)
    .tolist()
)

print("\nNCBI MRSN IDs:")

print(
    genome_df["MRSN"]
    .head(10)
    .tolist()
)


# ============================================================
# STEP 13 — FIND MRSN OVERLAP
# ============================================================

ncbi_mrsn = set(
    genome_df["MRSN"].dropna()
)

ast_mrsn = set(
    ast_df["MRSN"].dropna()
)

common_mrsn = (
    ncbi_mrsn &
    ast_mrsn
)

print(
    "\nMRSN IDs in NCBI:",
    len(ncbi_mrsn)
)

print(
    "MRSN IDs in AST:",
    len(ast_mrsn)
)

print(
    "Common MRSN IDs:",
    len(common_mrsn)
)


# ============================================================
# STEP 14 — MERGE
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 — MATCHING NCBI WITH AST")
print("=" * 70)

merged = pd.merge(
    genome_df,
    ast_df,
    on="MRSN",
    how="inner"
)


print(
    "\nGenome ↔ AST matches:",
    len(merged)
)


print("\nFirst 20 matches:")

print(
    merged[
        [
            "NCBI_Accession",
            "BioSample",
            "MRSN",
            "Cefepime"
        ]
    ].head(20).to_string(
        index=False
    )
)


# ============================================================
# STEP 15 — CREATE ML TARGET
# ============================================================

print("\n" + "=" * 70)
print("STEP 7 — CREATING ML TARGET")
print("=" * 70)

merged["Cefepime_Target"] = (
    merged["Cefepime"]
    .map({
        "R": 1,
        "S": 0
    })
)


print(
    """
R = Resistant = 1
S = Sensitive = 0
I = Intermediate
"""
)


# ============================================================
# STEP 16 — SAVE COMPLETE MAPPING
# ============================================================

merged.to_csv(
    MAPPING_OUTPUT,
    index=False
)

print(
    "\nComplete mapping saved:"
)

print(
    MAPPING_OUTPUT
)


# ============================================================
# STEP 17 — CREATE R/S DATASET
# ============================================================

rs_df = merged[
    merged["Cefepime"].isin(
        ["R", "S"]
    )
].copy()


# ============================================================
# STEP 18 — SAVE R/S DATASET
# ============================================================

rs_df.to_csv(
    RS_OUTPUT,
    index=False
)

print(
    "\nR/S-only mapping saved:"
)

print(
    RS_OUTPUT
)


# ============================================================
# STEP 19 — FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "\nTotal NCBI metadata records:",
    len(genome_df)
)

print(
    "Downloaded GCA genome folders:",
    len(gca_folders)
)

print(
    "AST isolates:",
    len(ast_df)
)

print(
    "Genome ↔ AST matches:",
    len(merged)
)

print(
    "R/S usable samples:",
    len(rs_df)
)


# ============================================================
# CEFEPIME DISTRIBUTION
# ============================================================

print(
    "\nCefepime distribution:"
)

print(
    merged[
        "Cefepime"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# FINAL R/S DISTRIBUTION
# ============================================================

print(
    "\nFinal R/S distribution:"
)

print(
    rs_df[
        "Cefepime"
    ].value_counts()
)


# ============================================================
# TARGET DISTRIBUTION
# ============================================================

print(
    "\nML target distribution:"
)

print(
    rs_df[
        "Cefepime_Target"
    ].value_counts()
)


# ============================================================
# AST WITHOUT NCBI GENOME
# ============================================================

missing_genome = ast_df[
    ~ast_df["MRSN"].isin(
        genome_df["MRSN"]
    )
].copy()


print(
    "\nAST isolates without NCBI metadata:",
    len(missing_genome)
)


missing_genome.to_csv(
    UNMATCHED_AST_OUTPUT,
    index=False
)


print(
    "Unmatched AST file saved:"
)

print(
    UNMATCHED_AST_OUTPUT
)


# ============================================================
# NCBI WITHOUT AST
# ============================================================

missing_ast = genome_df[
    ~genome_df["MRSN"].isin(
        ast_df["MRSN"]
    )
].copy()


print(
    "\nNCBI genomes without AST:",
    len(missing_ast)
)


# ============================================================
# FINISH
# ============================================================

print("\n" + "=" * 70)
print("PROCESS COMPLETED")
print("=" * 70)

print(
    """
The important files are:

1. genome_MRSN_Cefepime_mapping.csv
   → All NCBI ↔ AST matches

2. genome_MRSN_Cefepime_RS_only.csv
   → Only Resistant/Sensitive samples

3. AST_without_NCBI_genome.csv
   → AST isolates that do not have
     a matching NCBI metadata record
"""
)





CHECKING FILES
✓ NCBI metadata file found
✓ Excel AST file found

STEP 1 — READING NCBI METADATA
NCBI metadata records: 200

First 10 records:
 NCBI_Accession    BioSample       MRSN
GCA_019927265.1 SAMN18874781 MRSN761403
GCA_019927285.1 SAMN18874779 MRSN750999
GCA_019927305.1 SAMN18874777 MRSN742743
GCA_019927315.1 SAMN18874778 MRSN750877
GCA_019927325.1 SAMN18874775 MRSN736213
GCA_019927345.1 SAMN18874776 MRSN740795
GCA_019927385.1 SAMN18874774 MRSN731029
GCA_019927405.1 SAMN18874770 MRSN702261
GCA_019927425.1 SAMN18874771 MRSN702325
GCA_019927435.1 SAMN18874773 MRSN730567

STEP 2 — CHECKING DOWNLOADED GENOMES
Downloaded GCA genome folders: 57

First 10 downloaded genomes:
GCA_019927265.1
GCA_019927285.1
GCA_019927305.1
GCA_019927315.1
GCA_019927325.1
GCA_019927345.1
GCA_019927385.1
GCA_019927405.1
GCA_019927425.1
GCA_019927435.1

STEP 3 — READING AST EXCEL
Excel dimensions: (129, 177)

Excel header information:
Header row      : 1
MRSN column     : 0
Cefepime column : 24
MRSN heade

In [8]:
import os
import re
import pandas as pd

# ============================================================
# PATHS
# ============================================================

NCBI_METADATA = r"C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\assembly_data_report.jsonl"

AST_MAPPING = r"C:\Users\HP\Downloads\klebsiella_project\genome_MRSN_Cefepime_mapping.csv"

GENOME_DIR = r"C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data"

OUTPUT_DIR = r"C:\Users\HP\Downloads\klebsiella_project"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# STEP 1 — READ NCBI METADATA
# ============================================================

print("=" * 70)
print("STEP 1 — READING NCBI METADATA")
print("=" * 70)

ncbi = pd.read_json(NCBI_METADATA, lines=True)

print("NCBI records:", len(ncbi))


# ------------------------------------------------------------
# Find accession, BioSample and MRSN fields automatically
# ------------------------------------------------------------

def recursive_find(obj, keywords):
    """
    Search nested dictionaries/lists for values whose
    key contains one of the requested keywords.
    """
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():

            key_lower = str(key).lower()

            if any(k in key_lower for k in keywords):

                if isinstance(value, (str, int, float)):
                    results.append(str(value))

            results.extend(recursive_find(value, keywords))

    elif isinstance(obj, list):
        for item in obj:
            results.extend(recursive_find(item, keywords))

    return results


records = []

for _, row in ncbi.iterrows():

    row_dict = row.to_dict()

    # Convert entire row to searchable text
    text = str(row_dict)

    # Genome accession
    accession_matches = re.findall(
        r"GCA_\d+\.\d+|GCF_\d+\.\d+",
        text
    )

    # BioSample
    biosample_matches = re.findall(
        r"SAM[NED]\d+",
        text
    )

    # MRSN
    mrsn_matches = re.findall(
        r"MRSN\d+",
        text
    )

    records.append({
        "NCBI_Accession": accession_matches[0]
            if accession_matches else None,

        "BioSample": biosample_matches[0]
            if biosample_matches else None,

        "MRSN": mrsn_matches[0]
            if mrsn_matches else None
    })


ncbi_clean = pd.DataFrame(records)

ncbi_clean = ncbi_clean.dropna(
    subset=["NCBI_Accession", "MRSN"]
)

ncbi_clean = ncbi_clean.drop_duplicates(
    subset=["NCBI_Accession"]
)

print("\nClean NCBI records:", len(ncbi_clean))

print(ncbi_clean.head())


# ============================================================
# STEP 2 — FIND DOWNLOADED GENOMES
# ============================================================

print("\n" + "=" * 70)
print("STEP 2 — FINDING DOWNLOADED GENOMES")
print("=" * 70)

downloaded_genomes = []

for item in os.listdir(GENOME_DIR):

    full_path = os.path.join(GENOME_DIR, item)

    if os.path.isdir(full_path):

        if re.fullmatch(
            r"GCA_\d+\.\d+|GCF_\d+\.\d+",
            item
        ):
            downloaded_genomes.append(item)


downloaded_genomes = sorted(set(downloaded_genomes))

print(
    "Downloaded genome folders:",
    len(downloaded_genomes)
)

print("\nFirst 20:")

for genome in downloaded_genomes[:20]:
    print(genome)


# ============================================================
# STEP 3 — READ AST MAPPING
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — READING AST MAPPING")
print("=" * 70)

ast = pd.read_csv(AST_MAPPING)

print("Mapping records:", len(ast))

print("\nColumns:")
print(ast.columns.tolist())


# ============================================================
# STEP 4 — NORMALIZE MRSN
# ============================================================

def normalize_mrsn(x):

    if pd.isna(x):
        return None

    x = str(x).strip().upper()

    # Extract MRSN number
    match = re.search(r"MRSN\s*0*(\d+)", x)

    if match:
        return "MRSN" + match.group(1)

    return None


ncbi_clean["MRSN"] = ncbi_clean["MRSN"].apply(
    normalize_mrsn
)

ast["MRSN"] = ast["MRSN"].apply(
    normalize_mrsn
)


# ============================================================
# STEP 5 — NORMALIZE CEFEPIME
# ============================================================

ast["Cefepime"] = (
    ast["Cefepime"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("\nCefepime distribution:")

print(
    ast["Cefepime"].value_counts()
)


# ============================================================
# STEP 6 — KEEP ONLY R/S
# ============================================================

ast_rs = ast[
    ast["Cefepime"].isin(["R", "S"])
].copy()

ast_rs["Cefepime_Target"] = (
    ast_rs["Cefepime"]
    .map({
        "R": 1,
        "S": 0
    })
)

print("\nR/S AST isolates:", len(ast_rs))


# ============================================================
# STEP 7 — KEEP ONLY DOWNLOADED GENOMES
# ============================================================

downloaded_df = pd.DataFrame({
    "NCBI_Accession": downloaded_genomes
})

print("\nDownloaded genomes:", len(downloaded_df))


# ============================================================
# STEP 8 — MATCH NCBI + AST
# ============================================================

print("\n" + "=" * 70)
print("STEP 8 — MATCHING DOWNLOADED GENOMES WITH AST")
print("=" * 70)

# First restrict NCBI to downloaded genomes

ncbi_downloaded = ncbi_clean[
    ncbi_clean["NCBI_Accession"].isin(
        downloaded_genomes
    )
].copy()

print(
    "Downloaded genomes with NCBI metadata:",
    len(ncbi_downloaded)
)


# Merge using MRSN

matched = ncbi_downloaded.merge(
    ast_rs[
        ["MRSN", "Cefepime", "Cefepime_Target"]
    ],
    on="MRSN",
    how="inner"
)


# ============================================================
# STEP 9 — REMOVE DUPLICATES
# ============================================================

print("\nBefore duplicate removal:", len(matched))

matched = matched.drop_duplicates(
    subset=["NCBI_Accession"]
)

print(
    "After duplicate removal:",
    len(matched)
)


# ============================================================
# STEP 10 — FINAL TABLE
# ============================================================

final_columns = [
    "NCBI_Accession",
    "BioSample",
    "MRSN",
    "Cefepime",
    "Cefepime_Target"
]

final_ml = matched[
    final_columns
].copy()


# Sort

final_ml = final_ml.sort_values(
    "NCBI_Accession"
)


# ============================================================
# STEP 11 — DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL ML DATASET")
print("=" * 70)

print(
    final_ml.to_string(index=False)
)


# ============================================================
# STEP 12 — SAVE
# ============================================================

output_file = os.path.join(
    OUTPUT_DIR,
    "FINAL_ML_GENOME_CEFEPIME_DATASET.csv"
)

final_ml.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


# ============================================================
# STEP 13 — SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Downloaded genomes:",
    len(downloaded_genomes)
)

print(
    "Downloaded genomes with metadata:",
    len(ncbi_downloaded)
)

print(
    "Final labelled genomes:",
    len(final_ml)
)

print("\nCefepime labels:")

print(
    final_ml["Cefepime"].value_counts()
)

print("\nML target:")

print(
    final_ml["Cefepime_Target"].value_counts()
)

print("\nTarget meaning:")
print("0 = Sensitive")
print("1 = Resistant")


# ============================================================
# STEP 14 — CHECK GENOME FILES
# ============================================================

print("\n" + "=" * 70)
print("CHECKING GENOME SEQUENCE FILES")
print("=" * 70)

genome_files = []

for accession in final_ml["NCBI_Accession"]:

    folder = os.path.join(
        GENOME_DIR,
        accession
    )

    found = []

    if os.path.exists(folder):

        for root, dirs, files in os.walk(folder):

            for file in files:

                if file.endswith(
                    (".fna", ".fasta", ".fa")
                ):
                    found.append(
                        os.path.join(root, file)
                    )

    genome_files.append({
        "NCBI_Accession": accession,
        "Genome_Files": len(found)
    })


genome_check = pd.DataFrame(
    genome_files
)

print(
    genome_check.to_string(index=False)
)


# ============================================================
# SAVE GENOME + LABEL TABLE
# ============================================================

genome_label_file = os.path.join(
    OUTPUT_DIR,
    "GENOME_LABELS_FOR_CARD.csv"
)

final_ml.to_csv(
    genome_label_file,
    index=False
)

print("\n" + "=" * 70)
print("READY FOR CARD/RGI")
print("=" * 70)

print(
    genome_label_file
)



STEP 1 — READING NCBI METADATA
NCBI records: 200

Clean NCBI records: 200
    NCBI_Accession     BioSample        MRSN
0  GCF_019927265.1  SAMN18874781  MRSN761403
1  GCF_019927285.1  SAMN18874779  MRSN750999
2  GCF_019927305.1  SAMN18874777  MRSN742743
3  GCF_019927315.1  SAMN18874778  MRSN750877
4  GCF_019927325.1  SAMN18874775  MRSN736213

STEP 2 — FINDING DOWNLOADED GENOMES
Downloaded genome folders: 57

First 20:
GCA_019927265.1
GCA_019927285.1
GCA_019927305.1
GCA_019927315.1
GCA_019927325.1
GCA_019927345.1
GCA_019927385.1
GCA_019927405.1
GCA_019927425.1
GCA_019927435.1
GCA_019927465.1
GCA_019927485.1
GCA_019927505.1
GCA_019927525.1
GCA_019927535.1
GCA_019927565.1
GCA_019927585.1
GCA_019927605.1
GCA_019927625.1
GCA_019927645.1

STEP 3 — READING AST MAPPING
Mapping records: 200

Columns:
['NCBI_Accession', 'BioSample', 'MRSN', 'Cefepime', 'Cefepime_Target']

Cefepime distribution:
Cefepime
R    108
S     88
I      4
Name: count, dtype: int64

R/S AST isolates: 196

Downloaded genom

In [16]:
!docker --version

Docker version 28.3.2, build 578ccf6


In [17]:
!docker pull quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0

error during connect: Post "http://%2F%2F.%2Fpipe%2FdockerDesktopLinuxEngine/v1.51/images/create?fromImage=quay.io%2Fbiocontainers%2Frgi&tag=6.0.3--pyha8f3691_0": open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.


In [18]:
!docker desktop start

âœ“ Starting Docker Desktop


In [19]:
!docker info

Client:
 Version:    28.3.2
 Context:    desktop-linux
 Debug Mode: false
 Plugins:
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.6.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-ai.exe
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.25.0-desktop.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-buildx.exe
  cloud: Docker Cloud (Docker Inc.)
    Version:  v0.4.2
    Path:     C:\Program Files\Docker\cli-plugins\docker-cloud.exe
  compose: Docker Compose (Docker Inc.)
    Version:  v2.38.2-desktop.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-compose.exe
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.41
    Path:     C:\Program Files\Docker\cli-plugins\docker-debug.exe
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.1.11
    Path:     C:\Program Files\Docker\cli-plugins\docker-desktop.exe
  extension: Manages Docker extensions (Docker Inc.)
    Version:  v0.2.29
    Path:    

In [20]:
!docker run --rm hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world
4f55086f7dd0: Pulling fs layer
4f55086f7dd0: Download complete
4f55086f7dd0: Pull complete
Digest: sha256:5dd0d3e6e255913fc30f90b9f2b1d359cc2cbdb48090cc4b65f1676e203243cc
Status: Downloaded newer image for hello-world:latest


In [21]:
!docker pull quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0

6.0.3--pyha8f3691_0: Pulling from biocontainers/rgi
642efca944a0: Pulling fs layer
bd9ddc54bea9: Pulling fs layer
56b39463e514: Pulling fs layer
bd9ddc54bea9: Verifying Checksum
bd9ddc54bea9: Download complete
642efca944a0: Download complete
642efca944a0: Pull complete
bd9ddc54bea9: Pull complete
56b39463e514: Verifying Checksum
56b39463e514: Download complete
56b39463e514: Pull complete
Digest: sha256:27b9cd957831c075971e3f6bc52be4be5bbbae4f144a6240c7f5776bc0fa6f38
Status: Downloaded newer image for quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0
quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0


In [24]:
import os

genome_base = r"C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data"

genome_files = []

for root, dirs, files in os.walk(genome_base):
    for file in files:
        if file.lower().endswith((".fna", ".fa", ".fasta")):
            genome_files.append(os.path.join(root, file))

print("=" * 70)
print("GENOME FILE SEARCH")
print("=" * 70)

print("Genome files found:", len(genome_files))

print("\nFirst 10 genome files:")
for f in genome_files[:10]:
    print(f)

GENOME FILE SEARCH
Genome files found: 56

First 10 genome files:
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927265.1\GCA_019927265.1_ASM1992726v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927285.1\GCA_019927285.1_ASM1992728v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927305.1\GCA_019927305.1_ASM1992730v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927315.1\GCA_019927315.1_ASM1992731v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927325.1\GCA_019927325.1_ASM1992732v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927345.1\GCA_019927345.1_ASM1992734v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927385.1\GCA_019927385.1_ASM1992738v1_genomic.fna
C:\Users\HP\Downloads\klebsiella_dataset\ncbi_dataset\data\GCA_019927405.1\GCA_019927405.1_ASM1992740v1_genomic.fna
C:\Use

In [31]:
import subprocess

cmd = [
    "docker", "run", "--rm",
    "quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0",
    "rgi", "database", "--version"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)

STDOUT:
N/A

STDERR:

Return code: 0


In [32]:
import subprocess

cmd = [
    "docker", "run", "--rm",
    "quay.io/biocontainers/rgi:6.0.3--pyha8f3691_0",
    "bash", "-c",
    "rgi --version && echo '--- RGI LOCATION ---' && which rgi && echo '--- DATABASE LOCATIONS ---' && find / -name 'card.json' -o -name 'card.json*' 2>/dev/null | head -20"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)

STDOUT:

STDERR:
usage: rgi <command> [<args>]
            commands are:
               ---------------------------------------------------------------------------------------
               Database
               ---------------------------------------------------------------------------------------
               auto_load Automatically loads CARD database, annotations and k-mer database
               load      Loads CARD database, annotations and k-mer database
               clean     Removes BLAST databases and temporary files
               database  Information on installed card database
               galaxy    Galaxy project wrapper

               ---------------------------------------------------------------------------------------
               Genomic
               ---------------------------------------------------------------------------------------

               main     Runs rgi application
               tab      Creates a Tab-delimited from rgi results
       

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/msys2/win-64/current_repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/win-64/current_repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/noarch/current_repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/win-64/current_repodata.json HTTP/1.1" 304 0
DEBUG:url

Solving environment: ...working... unsuccessful attempt using repodata from current_repodata.json, retrying with next repodata source.
Solving environment: ...working... 
Found conflicts! Looking for incompatible packages.
This can take several minutes.  Press CTRL-C to abort.
failed
